# Step 1: CUDA profiling of our inference pipeline (cuda-profiling)

**Settings:** GPU **T4 x2** (we use one), Internet OFF is fine. **Input:** **llm-cls-train-v2**.

**Question:** our 10.6× speedup came from high-level tricks (fp16, sorting, 2 GPUs). Where does GPU time actually go now, and what's left to remove?

| Exp | What | Expectation (to be confirmed by data) |
|---|---|---|
| E0 | Time breakdown per batch: CPU collate vs GPU compute (CUDA events) | GPU-bound at bs=16 |
| E1 | torch.profiler trace: matmul vs attention vs elementwise kernels | Matmul-dominated; shows which attention kernel T4 uses |
| E2 | Pinned memory + async copy on a separate CUDA stream + no per-batch sync | Small gain if GPU-bound |
| E3 | torch.compile (kernel fusion of elementwise ops) | Some gain on elementwise share |
| E4 | Batch size 1 latency vs input length: eager vs **CUDA Graphs** | Big gain on short inputs (launch-bound), small on long |
| E5 | Why some batches fall back to unfused FP32 attention (found in E1) | Padding / forced backend reveals the cause |

Every experiment records PASS/FAIL; one failure doesn't stop the others. Output: `step1_results.csv`, `top_kernels.csv`, `trace_baseline.json` (open in Perfetto / chrome://tracing), `latency_bs1.png`.

In [ ]:
import os, glob, json, time, re, traceback
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def find(p):
    hits = glob.glob(f"/kaggle/input/**/{p}", recursive=True)
    assert hits, f"input not found: {p} (Add Input llm-cls-train-v2?)"
    return hits[0]

MODEL_DIR = os.path.dirname(find("qwen-cls/config.json"))
INFO = json.load(open(find("train_info.json")))
DEV = "cuda:0"
N_BENCH = 400          # val samples for E0/E2/E3 (25 batches of 16)
BS = 16

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
PAD_ID = tok.pad_token_id
val = pd.read_parquet(find("val.parquet"))

H = lambda s: tok(s, add_special_tokens=False)["input_ids"]
H_P, H_A, H_B, H_Q = H("Prompt: "), H("\n\nResponse A: "), H("\n\nResponse B: "), H("\n\nWhich response is better?")
def build(d):
    P = tok(d["prompt"].tolist(),     add_special_tokens=False, truncation=True, max_length=INFO["max_p"])["input_ids"]
    A = tok(d["response_a"].tolist(), add_special_tokens=False, truncation=True, max_length=INFO["max_r"])["input_ids"]
    B = tok(d["response_b"].tolist(), add_special_tokens=False, truncation=True, max_length=INFO["max_r"])["input_ids"]
    return [H_P + p + H_A + a + H_B + b + H_Q for p, a, b in zip(P, A, B)]

ids_all = build(val)
SUB = ids_all[:N_BENCH]
ORDER = np.argsort([-len(x) for x in SUB], kind="stable")          # length-sorted batching (our best setup)
BATCHES = [ORDER[i:i + BS] for i in range(0, len(SUB), BS)]

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, dtype=torch.float16,
                                                           device_map={"": DEV}).eval()
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
print("attention implementation:", getattr(model.config, "_attn_implementation", "?"))
print("bench samples:", len(SUB), "| batches:", len(BATCHES), "| mean len:", int(np.mean([len(x) for x in SUB])))

RESULTS = []
def rec(exp, status, **kv):
    RESULTS.append(dict(exp=exp, status=status, **kv))
    print(f"[{exp}] {status} {kv}")

def run_exp(name, fn):
    try:
        fn()
    except Exception as e:
        traceback.print_exc()
        rec(name, "FAIL", error=repr(e)[:300])
    torch.cuda.empty_cache()

## E0: per-batch time breakdown (baseline = our current best HF setup)
`collate` builds the padded batch on the CPU in Python and copies it to the GPU (pageable memory, blocking). The GPU sits idle during that time.

In [ ]:
def collate(seqs, device):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device)}

@torch.inference_mode()
def run_baseline(m, instrument=True):
    out = torch.zeros((len(SUB), 3))
    t_cpu = t_gpu = 0.0
    torch.cuda.synchronize(); start = time.perf_counter()
    for idx in BATCHES:
        t0 = time.perf_counter()
        batch = collate([SUB[j] for j in idx], DEV)
        t1 = time.perf_counter()
        e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        e0.record()
        logits = m(**batch, use_cache=False).logits
        e1.record()
        out[torch.as_tensor(idx)] = torch.softmax(logits.float(), -1).cpu()   # blocks until GPU done
        t_cpu += t1 - t0
        t_gpu += e0.elapsed_time(e1) / 1000
    wall = time.perf_counter() - start
    return out, wall, t_cpu, t_gpu

def e0():
    run_baseline(model)                                   # warmup
    probs, wall, t_cpu, t_gpu = run_baseline(model)
    global P_BASE, WALL_BASE
    P_BASE, WALL_BASE = probs, wall
    rec("E0_baseline", "PASS", samples_per_s=round(len(SUB) / wall, 2), wall_s=round(wall, 2),
        cpu_collate_pct=round(100 * t_cpu / wall, 1), gpu_busy_pct=round(100 * t_gpu / wall, 1))
run_exp("E0_baseline", e0)

## E1: torch.profiler: which kernels take the GPU time

In [ ]:
from torch.profiler import profile, ProfilerActivity

def self_dev_time(e):
    for a in ("self_device_time_total", "self_cuda_time_total"):
        if hasattr(e, a):
            return getattr(e, a)
    return 0.0

def category(name):
    n = name.lower()
    if any(k in n for k in ("fmha", "attention", "flash")):
        return "attention_fused"
    if "softmax_warp" in n or "sgemm" in n:            # fp32 bmm + softmax = SDPA math (unfused) fallback
        return "attention_math_fallback"
    if any(k in n for k in ("gemm", "cutlass", "cublas", "xmma", "s1688")):
        return "matmul"
    if "memcpy" in n or "memset" in n:
        return "memcpy"
    return "elementwise"

NON_KERNEL = ("Command Buffer Full", "Activity Buffer Request")

def e1():
    sub_batches = BATCHES[:6]
    with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for idx in sub_batches:
            batch = collate([SUB[j] for j in idx], DEV)
            logits = model(**batch, use_cache=False).logits
            torch.softmax(logits.float(), -1).cpu()
    prof.export_chrome_trace("/kaggle/working/trace_baseline.json")
    rows = []
    for e in prof.key_averages():
        t = self_dev_time(e)
        if t > 0 and not e.key.startswith("aten::") and e.key not in NON_KERNEL:   # kernel rows only
            rows.append(dict(kernel=e.key[:120], calls=e.count, self_gpu_ms=round(t / 1000, 2), category=category(e.key)))
    df = pd.DataFrame(rows).sort_values("self_gpu_ms", ascending=False)
    total = df["self_gpu_ms"].sum()
    df["pct"] = (100 * df["self_gpu_ms"] / total).round(1)
    df.to_csv("/kaggle/working/top_kernels.csv", index=False)
    display(df.head(15))
    share = df.groupby("category")["self_gpu_ms"].sum() / total * 100
    attn_kernels = df[df["category"].str.startswith("attention")]["kernel"].head(3).tolist()
    rec("E1_profile", "PASS", gpu_ms_total=round(total, 1),
        **{f"pct_{k.split('/')[0]}": round(v, 1) for k, v in share.items()},
        attention_kernels=attn_kernels)
run_exp("E1_profile", e1)

## E2: pinned memory + async copy on a side CUDA stream + no per-batch sync
While the GPU runs batch k, the CPU builds batch k+1 in **pinned** memory and copies it on a **separate CUDA stream**. Results stay on the GPU and are copied back once at the end.

In [ ]:
def collate_pinned(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.pin_memory(), mask.pin_memory()

@torch.inference_mode()
def run_overlap(m):
    copy_stream = torch.cuda.Stream()
    main = torch.cuda.current_stream()
    out = torch.zeros((len(SUB), 3), device=DEV)

    def prepare(k):
        ids_c, mask_c = collate_pinned([SUB[j] for j in BATCHES[k]])
        with torch.cuda.stream(copy_stream):
            ids_g = ids_c.to(DEV, non_blocking=True)
            mask_g = mask_c.to(DEV, non_blocking=True)
            ready = torch.cuda.Event()
            ready.record(copy_stream)
        return ids_g, mask_g, ready, (ids_c, mask_c)        # keep pinned tensors alive until copied

    torch.cuda.synchronize(); start = time.perf_counter()
    nxt = prepare(0)
    for k in range(len(BATCHES)):
        ids_g, mask_g, ready, _keep = nxt
        main.wait_event(ready)                               # compute waits only for its own copy
        ids_g.record_stream(main); mask_g.record_stream(main)
        logits = m(input_ids=ids_g, attention_mask=mask_g, use_cache=False).logits   # async launch
        if k + 1 < len(BATCHES):
            nxt = prepare(k + 1)                             # CPU work overlaps GPU compute
        out[torch.as_tensor(BATCHES[k], device=DEV)] = torch.softmax(logits.float(), -1)
    res = out.cpu()                                          # single sync at the end
    return res, time.perf_counter() - start

def e2():
    run_overlap(model)
    probs, wall = run_overlap(model)
    diff = float((probs - P_BASE).abs().max())
    rec("E2_pinned_stream", "PASS", samples_per_s=round(len(SUB) / wall, 2), wall_s=round(wall, 2),
        speedup_vs_E0=round(WALL_BASE / wall, 3), max_prob_diff=round(diff, 5))
run_exp("E2_pinned_stream", e2)

## E3: torch.compile (fuses small elementwise kernels: RMSNorm, SiLU, RoPE, adds)

In [ ]:
def e3():
    cm = torch.compile(model, dynamic=True)
    t = time.perf_counter()
    run_baseline(cm)                                         # first run = compilation
    compile_s = time.perf_counter() - t
    probs, wall, t_cpu, t_gpu = run_baseline(cm)
    diff = float((probs - P_BASE).abs().max())
    rec("E3_torch_compile", "PASS", samples_per_s=round(len(SUB) / wall, 2), wall_s=round(wall, 2),
        speedup_vs_E0=round(WALL_BASE / wall, 3), compile_s=round(compile_s, 1),
        gpu_busy_pct=round(100 * t_gpu / wall, 1), max_prob_diff=round(diff, 5))
run_exp("E3_torch_compile", e3)

## E4: batch size 1 latency vs input length: eager vs CUDA Graphs
Real-time serving scores **one** pair at a time. A forward pass launches hundreds of small kernels; each launch costs CPU time. A **CUDA Graph** records the whole forward once and replays it as a single launch. Expectation: big win on short inputs (launch-bound), small win on long inputs (compute-bound).

In [ ]:
LENGTHS = [32, 64, 128, 256, 512]
ITERS = 30
src = max(ids_all, key=len)                                  # longest real sample; truncate to each length

def time_it(fn):
    for _ in range(5):
        fn()
    torch.cuda.synchronize()
    ts = []
    for _ in range(ITERS):
        e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        e0.record(); fn(); e1.record()
        torch.cuda.synchronize()
        ts.append(e0.elapsed_time(e1))
    return float(np.percentile(ts, 50)), float(np.percentile(ts, 95))

def capture_graph(L):
    static_ids = torch.tensor([src[:L]], device=DEV)
    with torch.no_grad():
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):                               # warmup on a side stream before capture
                model(input_ids=static_ids, use_cache=False)
        torch.cuda.current_stream().wait_stream(s)
        g = torch.cuda.CUDAGraph()
        with torch.cuda.graph(g):
            static_out = model(input_ids=static_ids, use_cache=False).logits
    return g, static_ids, static_out

LAT = []
def e4():
    for L in LENGTHS:
        x = torch.tensor([src[:L]], device=DEV)
        with torch.no_grad():
            eager_fn = lambda: model(input_ids=x, use_cache=False).logits
            ref = eager_fn().float()
            p50_e, p95_e = time_it(eager_fn)
        row = dict(length=L, eager_p50_ms=round(p50_e, 2), eager_p95_ms=round(p95_e, 2))
        try:
            g, static_ids, static_out = capture_graph(L)
            static_ids.copy_(x)
            def graph_fn():
                g.replay()
                return static_out
            p50_g, p95_g = time_it(graph_fn)
            row.update(graph_p50_ms=round(p50_g, 2), graph_p95_ms=round(p95_g, 2),
                       speedup=round(p50_e / p50_g, 2),
                       max_logit_diff=round(float((static_out.float() - ref).abs().max()), 4))
            del g
        except Exception as e:
            row.update(graph_error=repr(e)[:200])
        LAT.append(row)
        print(row)
    df = pd.DataFrame(LAT)
    df.to_csv("/kaggle/working/latency_bs1.csv", index=False)
    display(df)
    ok = "speedup" in df and df["speedup"].notna().all()
    rec("E4_cuda_graphs_bs1", "PASS" if ok else "PARTIAL",
        speedup_short=df["speedup"].iloc[0] if ok else None,
        speedup_long=df["speedup"].iloc[-1] if ok else None)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["length"], df["eager_p50_ms"], "o-", label="eager")
    if "graph_p50_ms" in df:
        ax.plot(df["length"], df["graph_p50_ms"], "o-", label="CUDA Graph")
    ax.set_xscale("log", base=2); ax.set_xlabel("input tokens (batch size 1)")
    ax.set_ylabel("p50 latency (ms)"); ax.set_title(f"bs=1 latency: eager vs CUDA Graphs ({torch.cuda.get_device_name(0)})")
    ax.grid(alpha=0.3); ax.legend(); plt.tight_layout()
    plt.savefig("/kaggle/working/latency_bs1.png", dpi=150); plt.show()
run_exp("E4_cuda_graphs_bs1", e4)

## E5: why do some batches fall back to unfused FP32 attention?
E1 showed 2 of 6 batches (the longest) running attention as fp32 `bmm` + `softmax` (SDPA "math" backend) instead of the fused memory-efficient kernel, ~3× slower per layer. Here we (a) check every batch, unpadded vs padded to a multiple of 8 / 64 tokens, and (b) force the fused backend on a fallback batch so PyTorch tells us the reason.

In [ ]:
import warnings
from torch.nn.attention import sdpa_kernel, SDPBackend

def collate_pad(seqs, device, multiple):
    L = max(len(s) for s in seqs)
    L = ((L + multiple - 1) // multiple) * multiple
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device)}

def attention_path(batch):
    with torch.inference_mode(), profile(activities=[ProfilerActivity.CUDA]) as p:
        model(**batch, use_cache=False)
    names = [e.key.lower() for e in p.key_averages()]
    # fp32 softmax over the attention scores only appears in the unfused math path
    # (sgemm alone is not proof: the fp32 RoPE matmul also uses sgemm at long lengths)
    return any("fmha" in n for n in names), any("softmax_warp_forward<float, float, float, 10" in n for n in names)

def time_batch(batch, reps=3):
    ts = []
    with torch.inference_mode():
        model(**batch, use_cache=False)
        for _ in range(reps):
            e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            e0.record(); model(**batch, use_cache=False); e1.record()
            torch.cuda.synchronize(); ts.append(e0.elapsed_time(e1))
    return float(np.median(ts))

def e5():
    rows = []
    for k in range(min(8, len(BATCHES))):                    # longest batches first
        seqs = [SUB[j] for j in BATCHES[k]]
        for mult in (1, 8, 64):
            b = collate_pad(seqs, DEV, mult)
            fused, math = attention_path(b)
            rows.append(dict(batch=k, max_len=max(len(s) for s in seqs), padded_len=b["input_ids"].shape[1],
                             pad_multiple=mult, fused=fused, math_fallback=math, ms=round(time_batch(b), 1)))
    df = pd.DataFrame(rows)
    df.to_csv("/kaggle/working/attention_fallback.csv", index=False)
    display(df)

    reason, warn = "no fallback batch found", []
    fb = df[(df.pad_multiple == 1) & df.math_fallback]
    if len(fb):
        b = collate_pad([SUB[j] for j in BATCHES[int(fb.batch.iloc[0])]], DEV, 1)
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                with torch.inference_mode(), sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
                    model(**b, use_cache=False)
                reason = "forcing the fused kernel worked (fallback was a heuristic choice)"
            except Exception as e:
                reason = repr(e)[:300]
            warn = [str(x.message)[:250] for x in w]
        print("forced fused attention ->", reason)
        for m in warn[:5]:
            print("  warning:", m)

    tot = df.groupby("pad_multiple")["ms"].sum()
    rec("E5_attention_fallback", "PASS",
        fallback_batches={int(m): int(df[(df.pad_multiple == m)].math_fallback.sum()) for m in (1, 8, 64)},
        total_ms={int(m): round(v, 1) for m, v in tot.items()},
        forced_fused=reason, warnings=warn[:3])
run_exp("E5_attention_fallback", e5)

## Summary

In [ ]:
res = pd.DataFrame(RESULTS)
res.to_csv("/kaggle/working/step1_results.csv", index=False)
pd.set_option("display.max_colwidth", 100)
display(res)